# RAG_03: Optuna Weight Optimization

Find the optimal weight vector `[α_w1, α_w2, α_w3, α_w5, α_w10, α_summary]`
that maximizes retrieval quality as judged by GPT-4.1.

**Input:** `data/rag_embeddings.npz`, `data/rag_metadata.parquet`
**Output:**
- `data/rag_optimal_weights.json` — best weights
- `data/rag_optuna_study.db` — SQLite study (resumable)
- `data/rag_judge_cache.jsonl` — cached LLM judge ratings

In [ ]:
import json
import numpy as np
import pandas as pd
import optuna
from pathlib import Path
from tqdm.notebook import tqdm

# ============================================================
# Configuration
# ============================================================
JUDGE_CACHE_FILE = Path("data/rag_judge_cache.jsonl")
STUDY_DB = "sqlite:///data/rag_optuna_study.db"
OUTPUT_WEIGHTS_FILE = Path("data/rag_optimal_weights.json")

N_EVAL_STATES = 80       # number of evaluation query states
N_OPTUNA_TRIALS = 50
TOP_K = 5                # retrieve top-k per query

## Load Index & Retrieval Functions

In [ ]:
from rag_retrieval import (
    load_rag_index, retrieve_for_state_index,
    WINDOW_NAMES, uniform_weights,
)

embedding_store, metadata_df = load_rag_index()
active_windows = list(embedding_store.keys())
print(f"Loaded {len(metadata_df)} states from {metadata_df['conversation_id'].nunique()} conversations")
print(f"Active windows in this index: {active_windows}")

## Sample Evaluation States

Stratified by conversation position (early/mid/late) and conversation length
(short/medium/long). Excludes single-state conversations.

In [ ]:
def sample_eval_states(metadata_df: pd.DataFrame, n: int = N_EVAL_STATES,
                       seed: int = 42) -> list[int]:
    """Sample evaluation states stratified by position and conversation length."""
    rng = np.random.RandomState(seed)

    # Count states per conversation
    conv_sizes = metadata_df.groupby('conversation_id').size()

    # Exclude conversations with only 1 state
    valid_convs = conv_sizes[conv_sizes > 1].index
    df = metadata_df[metadata_df['conversation_id'].isin(valid_convs)].copy()

    # Add normalized position within each conversation (0=early, 1=late)
    df['_max_state'] = df.groupby('conversation_id')['state_index'].transform('max')
    df['_position'] = df['state_index'] / df['_max_state'].clip(lower=1)

    # Position tercile: early (0-0.33), mid (0.33-0.67), late (0.67-1.0)
    df['_pos_bin'] = pd.cut(df['_position'], bins=[-.01, 0.33, 0.67, 1.01],
                            labels=['early', 'mid', 'late'])

    # Conversation length tercile
    df['_conv_len'] = df.groupby('conversation_id')['state_index'].transform('count')
    df['_len_bin'] = pd.qcut(df['_conv_len'], q=3, labels=['short', 'medium', 'long'],
                             duplicates='drop')

    # Stratified sample: try to get equal counts from each (position, length) bucket
    df['_stratum'] = df['_pos_bin'].astype(str) + '_' + df['_len_bin'].astype(str)
    n_strata = df['_stratum'].nunique()
    per_stratum = max(1, n // n_strata)

    sampled = []
    for _, group in df.groupby('_stratum'):
        take = min(per_stratum, len(group))
        chosen = group.sample(n=take, random_state=rng)
        sampled.append(chosen)

    result = pd.concat(sampled)

    # If we have fewer than n, top up randomly from remaining
    if len(result) < n:
        remaining = df[~df.index.isin(result.index)]
        extra = min(n - len(result), len(remaining))
        result = pd.concat([result, remaining.sample(n=extra, random_state=rng)])

    # Trim if over
    if len(result) > n:
        result = result.sample(n=n, random_state=rng)

    return result.index.tolist()


eval_indices = sample_eval_states(metadata_df, n=N_EVAL_STATES)
print(f"Sampled {len(eval_indices)} evaluation states")

# Distribution check
eval_df = metadata_df.iloc[eval_indices]
print(f"Position distribution (by n_turns_available quartile):")
print(pd.qcut(eval_df['n_turns_available'], q=3, duplicates='drop').value_counts())

## LLM Judge with Disk-Backed Cache

In [ ]:
from safechain.lcel import model

judge_model = model('gpt-4.1')

JUDGE_PROMPT_TEMPLATE = """You are evaluating whether two sales call states are similar enough that the same agent response strategy would be appropriate for both.

State A:
{state_a_context}

State B:
{state_b_context}

Rate the similarity on a 1-5 scale:
1 - Completely different situations, same response would be inappropriate
2 - Some surface similarity but different underlying dynamics
3 - Moderately similar, same general approach might work but details differ
4 - Very similar, the same response strategy would likely be effective
5 - Nearly identical situations

Respond with only the number."""


def _state_context(row: pd.Series) -> str:
    """Build the judge-facing context for a single state.

    Always includes recent turns (w3). Prefers the structured GPT-4.1 summary
    when available; falls back to full history when running with
    USE_SUMMARY=False.
    """
    parts = [f"Recent turns:\n{row['w3_text']}"]
    if 'w_summary_text' in row.index and pd.notna(row['w_summary_text']):
        parts.append(f"Summary:\n{row['w_summary_text']}")
    elif 'w_full_text' in row.index and pd.notna(row['w_full_text']):
        parts.append(f"Full context:\n{row['w_full_text']}")
    return "\n\n".join(parts)


def load_judge_cache(cache_path: Path) -> dict[str, int]:
    """Load existing judge ratings from JSONL cache."""
    cache = {}
    if cache_path.exists():
        with open(cache_path, 'r', encoding='utf-8') as f:
            for line in f:
                entry = json.loads(line)
                cache[entry['key']] = entry['rating']
    return cache


def save_judge_rating(cache_path: Path, key: str, rating: int,
                      query_idx: int, candidate_idx: int):
    """Append one rating to the cache file (incremental, crash-safe)."""
    with open(cache_path, 'a', encoding='utf-8') as f:
        f.write(json.dumps({
            'key': key, 'rating': rating,
            'query_idx': query_idx, 'candidate_idx': candidate_idx,
        }) + '\n')


def judge_similarity(query_idx: int, candidate_idx: int,
                     cache: dict[str, int]) -> int:
    """Rate similarity between two states using GPT-4.1, with caching."""
    key = f"{query_idx}_{candidate_idx}"
    if key in cache:
        return cache[key]

    q = metadata_df.iloc[query_idx]
    c = metadata_df.iloc[candidate_idx]

    prompt = JUDGE_PROMPT_TEMPLATE.format(
        state_a_context=_state_context(q),
        state_b_context=_state_context(c),
    )

    response = judge_model.invoke(prompt)
    if hasattr(response, 'content'):
        response = response.content

    try:
        rating = int(str(response).strip())
        rating = max(1, min(5, rating))
    except ValueError:
        rating = 3  # fallback for unexpected output

    cache[key] = rating
    save_judge_rating(JUDGE_CACHE_FILE, key, rating, query_idx, candidate_idx)
    return rating


# Load existing cache
judge_cache = load_judge_cache(JUDGE_CACHE_FILE)
print(f"Judge cache: {len(judge_cache)} ratings loaded")

## Optuna Objective Function

In [ ]:
def objective(trial: optuna.Trial) -> float:
    """Optuna objective: propose weights, retrieve, judge, return mean score."""
    # Propose one raw weight per active window (adapts to USE_SUMMARY)
    raw_weights = {
        name: trial.suggest_float(f'w_{name}', 0.01, 10.0)
        for name in active_windows
    }

    # Normalize to sum to 1
    total = sum(raw_weights.values())
    weights = {name: v / total for name, v in raw_weights.items()}

    all_ratings = []
    for q_idx in eval_indices:
        results = retrieve_for_state_index(
            q_idx, embedding_store, metadata_df,
            weights=weights, k=TOP_K,
        )
        for c_idx in results.index:
            rating = judge_similarity(q_idx, c_idx, judge_cache)
            all_ratings.append(rating)

    mean_score = np.mean(all_ratings)

    # Log normalized weights for inspection
    for name in active_windows:
        trial.set_user_attr(f'norm_w_{name}', round(weights[name], 4))
    trial.set_user_attr('n_ratings', len(all_ratings))
    trial.set_user_attr('cache_size', len(judge_cache))

    return mean_score

## Run Optuna Study

Uses SQLite storage for resumability — if interrupted, re-run this cell to continue.

In [ ]:
study = optuna.create_study(
    direction='maximize',
    study_name='rag_weight_optimization',
    storage=STUDY_DB,
    load_if_exists=True,
)

print(f"Study has {len(study.trials)} existing trials")
remaining = max(0, N_OPTUNA_TRIALS - len(study.trials))

if remaining > 0:
    print(f"Running {remaining} more trials...")
    study.optimize(objective, n_trials=remaining)
else:
    print("All trials already complete.")

## Results: Best Weights

In [ ]:
best_trial = study.best_trial
best_weights = {
    name: best_trial.user_attrs[f'norm_w_{name}']
    for name in active_windows
}

print(f"Best mean judge score: {best_trial.value:.3f} (trial #{best_trial.number})")
print(f"\nOptimal weights (sorted by importance):")
for name, w in sorted(best_weights.items(), key=lambda x: -x[1]):
    bar = "#" * int(w * 50)
    print(f"  {name:>10}: {w:.4f}  {bar}")

# Save
Path("data").mkdir(exist_ok=True)
with open(OUTPUT_WEIGHTS_FILE, 'w') as f:
    json.dump(best_weights, f, indent=2)
print(f"\nSaved to {OUTPUT_WEIGHTS_FILE}")

## Optuna Visualization

In [ ]:
from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances,
    plot_parallel_coordinate,
)

plot_optimization_history(study).show()
plot_param_importances(study).show()
plot_parallel_coordinate(study).show()

## Held-Out Validation

Sample 30 new states (not in eval set), compare optimized weights vs uniform baseline.

In [ ]:
# Sample holdout states (different seed, exclude eval set)
holdout_indices = sample_eval_states(metadata_df, n=30, seed=99)
holdout_indices = [i for i in holdout_indices if i not in set(eval_indices)]
print(f"Holdout states: {len(holdout_indices)}")


def score_weights(indices, weights, label=""):
    """Run retrieval + judge for a list of query indices, return mean score."""
    ratings = []
    for q_idx in tqdm(indices, desc=label):
        results = retrieve_for_state_index(
            q_idx, embedding_store, metadata_df,
            weights=weights, k=TOP_K,
        )
        for c_idx in results.index:
            rating = judge_similarity(q_idx, c_idx, judge_cache)
            ratings.append(rating)
    return np.mean(ratings), ratings


baseline_weights = uniform_weights(active_windows)
opt_mean, opt_ratings = score_weights(holdout_indices, best_weights, "Optimized")
uni_mean, uni_ratings = score_weights(holdout_indices, baseline_weights, "Uniform")

print(f"\nHeld-out validation ({len(holdout_indices)} states, {TOP_K} matches each):")
print(f"  Optimized weights: {opt_mean:.3f} mean score")
print(f"  Uniform weights:   {uni_mean:.3f} mean score")
print(f"  Improvement:       {opt_mean - uni_mean:+.3f}")
print(f"\nRating distribution (optimized):")
print(pd.Series(opt_ratings).value_counts().sort_index())

## Failure Analysis

Show cases where optimized retrieval scored poorly (rating <= 2). These reveal
systematic blind spots in the embedding approach.

In [ ]:
# Collect all rated pairs from the holdout validation
poor_matches = []
for q_idx in holdout_indices:
    results = retrieve_for_state_index(
        q_idx, embedding_store, metadata_df,
        weights=best_weights, k=TOP_K,
    )
    for c_idx in results.index:
        key = f"{q_idx}_{c_idx}"
        rating = judge_cache.get(key)
        if rating is not None and rating <= 2:
            poor_matches.append((q_idx, c_idx, rating))

print(f"Poor matches (rating <= 2): {len(poor_matches)}")
print("-" * 80)


def _secondary_context(row: pd.Series) -> str:
    """Pick the richer context available: summary if present, else w_full."""
    if 'w_summary_text' in row.index and pd.notna(row['w_summary_text']):
        return f"summary: {row['w_summary_text'][:120]}"
    if 'w_full_text' in row.index and pd.notna(row['w_full_text']):
        return f"full:    {row['w_full_text'][:120]}"
    return ""


for q_idx, c_idx, rating in poor_matches[:5]:
    q = metadata_df.iloc[q_idx]
    c = metadata_df.iloc[c_idx]
    print(f"\nQuery:  conv={q['conversation_id']}, state={q['state_index']}, "
          f"turns={q['n_turns_available']}")
    print(f"  w1: {q['w1_text'][:120]}")
    print(f"  {_secondary_context(q)}")
    print(f"Match:  conv={c['conversation_id']}, state={c['state_index']} "
          f"(rating={rating})")
    print(f"  w1: {c['w1_text'][:120]}")
    print(f"  {_secondary_context(c)}")